In [1]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
from unsloth import FastLanguageModel 
from unsloth import is_bfloat16_supported
#"meta-llama/Llama-3.1-70B-Instruct"
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-70B-Instruct",device_map="auto")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-70B-Instruct",device_map="auto")
# Use a pipeline as a high-level helper
# unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit

model,tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

tokenizer.pad_token = tokenizer.eos_token

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

/data/jiacheng/miniconda3/envs/common/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
==((====))==  Unsloth 2024.11.3: Fast Llama patching. Transformers = 4.46.2.
   \\   /|    GPU: NVIDIA H100 80GB HBM3. Max memory: 79.097 GB. Platform = Linux.
O^O/ \_/ \    Pytorch: 2.4.1+cu121. CUDA = 9.0. CUDA Toolkit = 12.1.
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.28.post1. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards: 100%|██████████| 6/6 [00:08<00:00,  1.42s/it]


In [6]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference


LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 8192, padding_idx=128004)
    (layers): ModuleList(
      (0-79): 80 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear4bit(in_features=8192, out_features=8192, bias=False)
          (k_proj): Linear4bit(in_features=8192, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=8192, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=8192, out_features=8192, bias=False)
          (rotary_emb): LlamaExtendedRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear4bit(in_features=8192, out_features=28672, bias=False)
          (up_proj): Linear4bit(in_features=8192, out_features=28672, bias=False)
          (down_proj): Linear4bit(in_features=28672, out_features=8192, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((8192,), eps=1e-05)
        (post_attention_layernorm): L

In [7]:
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)

In [ ]:
pipe

In [ ]:
import torch
torch.__version__



In [3]:
messages=[
                    {"role": "system", "content": """
Prompt:

You are given a list of pairs in the format [A, B], which will always include the [SAME ENTITY]. Your task is to reorganize these pairs such that the more general or abstract concept comes first, followed by the more specific concept. Here are the rules to follow:

Swap the order of [A, B] to [B, A] if A is more specific and B is more general. B may be larger than A in terms of scope or applicability.
Keep the original order [A, B] if both elements are of the same level of specificity or already correctly ordered.
For notable entities or specific items within a broader category, ensure the broader category comes first, swapping if necessary.
Examples:

[BEIJING, CHINA] should become [CHINA, BEIJING].
[SHANGHAI, CHINA] should become [CHINA, SHANGHAI].
[CHINA, EAST ASIA] should become [EAST ASIA, CHINA].
[BEIJING, BEIJING UNIVERSITY] should remain [BEIJING, BEIJING UNIVERSITY].
[TIANANMEN, BEIJING] should become [BEIJING, TIANANMEN].
[iPhone, Apple] should become [Apple, iPhone].
[Diseases, Symptoms] should become [Symptoms, Diseases].
[Body Location, Diseases] should become [Diseases, Body Location].
[Treatment Drugs, Diseases] should become [Diseases, Treatment Drugs].
[Treatment, Diseases] should become [Diseases, Treatment].
[attack technique, Software] should become [Software, attack technique].
[mitigation technique, attack technique] should become [attack technique, mitigation technique].

[math, mathematics department] should remain [math, mathematics department].
Apply these rules consistently to transform the list of pairs, ensuring that the more general concept (B) precedes the specific concept (A) unless both are of equal specificity or already correctly ordered.

Please return in the following JSON format:

if [SAME ENTITY] is the source(first of the list), put it in as_source, if [SAME ENTITY] is the target(second of the list), put it in as_target.  Make sure the order of each relationship is the correctly ordered one instead of the originial one.

First, give some analysis, and then return the following example json between OUTPUT_START and OUTPUT_END:

<JSON example>
{
    "as_source": [[SAME ENTITY, B],[SAME ENTITY, C],[SAME ENTITY, D]],
    "as_target": [[E, SAME ENTITY],[F, SAME ENTITY],[G, SAME ENTITY]]
}


"""},
                    {"role": "user", "content": """The following relationships are given: [["Beijing","China"],["Beijing","Forbidden City"]]" + " The [SAME ENTITY]  is [Beijing]"""},
                ]

In [22]:
content = pipe(messages, max_length=10000, do_sample=True, temperature=0.1)

In [23]:
content[0]["generated_text"][-1]["content"]

'Analysis:\n\nThe given relationships are [["Beijing","China"],["Beijing","Forbidden City"]]. The [SAME ENTITY] is [Beijing]. \n\nTo reorganize these pairs, we need to apply the rules:\n\n- For the pair ["Beijing","China"], since "China" is a more general concept (a country) and "Beijing" is a more specific concept (a city), we should swap the order to ["China","Beijing"].\n- For the pair ["Beijing","Forbidden City"], since "Beijing" is a broader category (a city) and "Forbidden City" is a notable entity within that category (a specific location), we should swap the order to ["Beijing","Forbidden City"] is incorrect and should remain as ["Forbidden City","Beijing"] is incorrect and should be ["Beijing","Forbidden City"].\n\nHowever, Forbidden City is a notable entity within Beijing, so the correct order should be ["Beijing","Forbidden City"].\n\nOUTPUT_START\n{\n    "as_source": [["Beijing","Forbidden City"]],\n    "as_target": [["China","Beijing"]]\n}\nOUTPUT_END'

In [24]:
content_short = content[0]["generated_text"][-1]["content"]

In [25]:
content_short = content_short.split('OUTPUT_START', 1)[-1].rsplit('OUTPUT_END', 1)[0]

In [ ]:
content_short

'\n{\n    "as_source": [["Beijing","Forbidden City"]],\n    "as_target": [["China","Beijing"]]\n}\n'

: 

In [ ]:
import json
content2 = json.loads(content_short)

In [ ]:
content2

: 

In [ ]:
content2